# EarthBridge Kaggle Baseline Training

Use this notebook for free online training on the Kaggle BEN-14K dataset. It expects `/kaggle/input/datasets/narendraaironi/bigearthnet-14k/BEN_14k`, then exports `artifacts/earthbridge_export.zip` for the local demo.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = ""  # Optional: set to your GitHub repo URL.
ROOT = Path("/kaggle/working/EarthBridge")

if REPO_URL and not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

if not ROOT.exists():
    ROOT = Path.cwd()

os.chdir(ROOT)
print("Working directory:", ROOT)

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q faiss-cpu rasterio albumentations opencv-python matplotlib tensorboard fastapi "uvicorn[standard]" python-multipart
!python -m pip install -q -e .

## Configure Dataset

The default path targets Kaggle BEN-14K. `BigEarthNet-S1` is treated as SAR and `BigEarthNet-S2` is treated as multispectral.

In [ ]:
DATA_RAW = Path("/kaggle/input/datasets/narendraaironi/bigearthnet-14k/BEN_14k")
IMAGE_ROOT = DATA_RAW.parent
LEFT_MODALITY = "multispectral"
RIGHT_MODALITY = "sar"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DATA_RAW:", DATA_RAW)
print("IMAGE_ROOT:", IMAGE_ROOT)
print("PAIR:", LEFT_MODALITY, "->", RIGHT_MODALITY)
print("DEVICE:", DEVICE)
assert DATA_RAW.exists(), f"Dataset path does not exist: {DATA_RAW}"

In [ ]:
!python scripts/inspect_data.py \
  --input "{DATA_RAW}" \
  --output-dir artifacts/reports
!python scripts/build_manifest.py \
  --input "{DATA_RAW}" \
  --output data/manifests/samples.csv
!python scripts/create_splits.py \
  --manifest data/manifests/samples.csv \
  --output-dir data/manifests


In [ ]:
!python scripts/run_tiny_overfit_matrix.py \
  --manifest data/manifests/train.csv \
  --root-dir "{IMAGE_ROOT}" \
  --output-dir artifacts/tiny_overfit \
  --left-modality "{LEFT_MODALITY}" \
  --right-modality "{RIGHT_MODALITY}" \
  --pair-count 128 \
  --image-size 128 \
  --batch-size 128 \
  --epochs 100 \
  --num-workers 8 \
  --device "{DEVICE}"
BEST_CONFIG = Path("artifacts/tiny_overfit/best_tiny_overfit_config.json")
assert BEST_CONFIG.exists(), "Tiny real-data overfit gate failed; inspect artifacts/tiny_overfit/tiny_overfit_matrix_report.json before full training."


In [ ]:
best = json.loads(BEST_CONFIG.read_text())
cfg = best["config"]
command = [
    sys.executable, "scripts/run_cloud_pipeline.py",
    "--data-raw", str(DATA_RAW),
    "--image-root", str(IMAGE_ROOT),
    "--left-modality", LEFT_MODALITY,
    "--right-modality", RIGHT_MODALITY,
    "--image-size", str(cfg["image_size"]),
    "--batch-size", "128",
    "--epochs", "25",
    "--num-workers", "8",
    "--validation-every", "5",
    "--validation-pair-limit", "512",
    "--learning-rate", str(cfg["learning_rate"]),
    "--temperature", str(cfg["temperature"]),
    "--projection-dropout", "0",
    "--semantic-loss-weight", "0",
    "--hard-negative-loss-weight", "0",
    "--diagnostic-sample-count", "128",
    "--seed", str(cfg["seed"]),
    "--device", DEVICE,
    "--top-k", "10",
    "--export-zip", "artifacts/earthbridge_export.zip",
]
if cfg.get("learnable_temperature"):
    command.append("--learnable-temperature")
if DEVICE == "cuda":
    command.append("--mixed-precision")
subprocess.run(command, check=True)
print("Download this file from Kaggle:", ROOT / "artifacts/earthbridge_export.zip")